# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data00077/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Random Forest as the main learned model because this is a binary classification and ranking problem: the goal is to identify pages that are more likely to be declining and prioritize them for review.

Random Forest can capture non-linear relationships between the available page-level signals without requiring a linear relationship between each feature and the outcome. I will compare it with the Week-4 rule baseline using the same test split and the same ranking metrics.

I will use Precision@50 as the main comparison metric because the practical question is whether the top 50 pages selected by the model contain a higher share of declining pages. I will also report Precision@20 and Precision@100 so that the result is not reduced to a single score.




In [12]:
%cd /content/flyrank-ml-internship

!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py

!find . -type f | grep -E "feature|baseline|refresh"

/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
./work/notebooks/w04_baseline_score.ipynb
./work/notebooks/w03_feature_leakage_check.ipynb
./skills/building-baselines/SKILL.md
./outputs/charts/top_feature_importance.svg
./outputs/refresh_queue_sample.csv
./data/processed/feature_metadata.json
./data/processed/baseline_refresh_queue.csv
./data/processed/baseline_metadata.json
./data/processed/refresh_feature_vector.csv
./data/raw/content_refresh_anonymized.csv
./scripts/02_baseline_score.py
./scripts/01_prepare_features.py


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-aware holdout when the data supports it. Pages from selected clients are kept entirely in the test set so that the same client does not appear in both training and testing.

This is more honest for the question because the model should be evaluated on pages from clients it did not train on. The split uses a fixed random seed of 42 for reproducibility. If a valid client holdout cannot be formed, the pipeline falls back to a stratified row-level holdout.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
# ML-08 — Train and compare against Week-4 baseline

import sys
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Make repo scripts importable
REPO_ROOT = Path("/content/flyrank-ml-internship")
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k,
)

FEATURE_PATH = REPO_ROOT / "data/processed/refresh_feature_vector.csv"
BASELINE_PATH = REPO_ROOT / "data/processed/baseline_refresh_queue.csv"

RANDOM_STATE = 42

# -----------------------------
# 1. Load data
# -----------------------------
frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)

print("Rows:", len(frame))
print("Features available:", len(frame.columns))
print("Positive labels:", int(frame["is_declining_label"].sum()))
print("Positive rate:", round(frame["is_declining_label"].mean(), 4))

# -----------------------------
# 2. Build feature matrix
# -----------------------------
numeric_features = [
    c for c in MODEL_NUMERIC_FEATURES
    if c in frame.columns
]

categorical_features = [
    c for c in MODEL_CATEGORICAL_FEATURES
    if c in frame.columns
]

numeric_frame = (
    frame[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

categorical_frame = (
    frame[categorical_features]
    .fillna("unknown")
    .astype(str)
)

encoded_frame = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float,
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True),
    ],
    axis=1,
)

y = frame["is_declining_label"].astype(int)

print("Final model feature count:", X.shape[1])

# -----------------------------
# 3. Same client-aware split
# -----------------------------
all_indices = np.arange(len(frame))

client_series = (
    frame["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = client_series.drop_duplicates().to_numpy()

if len(unique_clients) >= 5:

    rng = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = rng.permutation(unique_clients)

    test_client_count = max(
        1,
        int(round(len(shuffled_clients) * 0.20))
    )

    test_clients = set(
        shuffled_clients[:test_client_count]
    )

    test_mask = client_series.isin(test_clients).to_numpy()

    train_idx = all_indices[~test_mask]
    test_idx = all_indices[test_mask]

    valid_client_split = (
        len(train_idx) > 0
        and len(test_idx) > 0
        and y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    )

    if valid_client_split:
        split_strategy = "client_holdout"

    else:
        train_idx, test_idx = train_test_split(
            all_indices,
            test_size=0.20,
            random_state=RANDOM_STATE,
            stratify=y,
        )

        split_strategy = "stratified_row_holdout"

else:

    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    split_strategy = "stratified_row_holdout"


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Split strategy:", split_strategy)
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

# -----------------------------
# 4. Baseline on SAME test rows
# -----------------------------
baseline_lookup = (
    baseline_frame
    .set_index("content_id")["baseline_refresh_score"]
)

baseline_scores = (
    frame.iloc[test_idx]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

# -----------------------------
# 5. Models
# -----------------------------
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

# -----------------------------
# 6. Evaluation helper
# -----------------------------
def evaluate_scores(y_true, scores):

    predictions = (
        np.asarray(scores) >= 0.5
    ).astype(int)

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Precision@20": precision_at_k(
            y_true,
            scores,
            20
        ),

        "Precision@50": precision_at_k(
            y_true,
            scores,
            50
        ),

        "Precision@100": precision_at_k(
            y_true,
            scores,
            100
        ),

        "ROC-AUC": roc_auc_score(
            y_true,
            scores
        ),

        "Average Precision": average_precision_score(
            y_true,
            scores
        ),
    }


# -----------------------------
# 7. Evaluate baseline
# -----------------------------
results = []

baseline_result = evaluate_scores(
    y_test,
    baseline_scores
)

results.append({
    "Method": "Week-4 Baseline",
    **baseline_result
})

# -----------------------------
# 8. Train models
# -----------------------------
trained_models = {}

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(
        X_train,
        y_train
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    trained_models[name] = model

    model_result = evaluate_scores(
        y_test,
        probabilities
    )

    results.append({
        "Method": name,
        **model_result
    })


# -----------------------------
# 9. Final comparison table
# -----------------------------
comparison = pd.DataFrame(results)

print("\nMODEL vs BASELINE")
display(
    comparison.round(4)
)

print(
    "\nBase rate:",
    round(y_test.mean(), 4)
)

print(
    "\nBest Precision@50:",
    comparison.loc[
        comparison["Precision@50"].idxmax(),
        "Method"
    ]
)

Rows: 30000
Features available: 52
Positive labels: 16262
Positive rate: 0.5421
Final model feature count: 52
Split strategy: client_holdout
Train rows: 27675
Test rows: 2325

Training: Logistic Regression

Training: Decision Tree

Training: Random Forest

MODEL vs BASELINE


,Method,Accuracy,Precision,Recall,F1,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision
0,Week-4 Baseline,0.6086,0.4986,0.1892,0.2743,0.15,0.24,0.36,0.6269,0.4676
1,Logistic Regression,0.6606,0.5659,0.5666,0.5662,0.35,0.40,0.44,0.7003,0.5215
2,Decision Tree,0.6766,0.5686,0.7162,0.6339,0.45,0.58,0.62,0.7415,0.5753
3,Random Forest,0.6723,0.5610,0.7437,0.6395,0.65,0.74,0.72,0.7500,0.6182



Base rate: 0.391

Best Precision@50: Random Forest


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest produced 233 false negatives and 529 false positives on the test holdout, with 1,563 predictions classified as correct.

Three reviewed false negatives were:
- content_28b4223f4e5f: observed declining label = 1, model probability = 0.080.
- content_34b14c00f80c: observed declining label = 1, model probability = 0.082.
- content_79ac977c6e0b: observed declining label = 1, model probability = 0.150.

These cases were difficult because the available features did not produce a strong enough signal for the model to rank them highly, even though the observed label was declining.

Three reviewed false positives were:
- content_d2dffcc697a4: observed label = 0, model probability = 0.737.
- content_00603b0349b4: observed label = 0, model probability = 0.735.
- content_331182ca4: observed label = 0, model probability = 0.734.

All three false positives came from client_f74efabef1. This suggests that some client-specific patterns may resemble declining cases without matching the observed label. Because the available analysis does not establish causality, this should be treated as an observed error pattern rather than a causal explanation.

The most important Random Forest features were days_with_impressions, log_impressions_90d, avg_position, and content_age_days. These features plausibly relate to visibility, search position, and content maturity. Feature importance represents association rather than causation.

Overall, Random Forest substantially improved ranking performance over the Week-4 baseline. Precision@50 was 0.74 compared with 0.24 for the baseline. Precision@20 was 0.65 versus 0.15, and Precision@100 was 0.72 versus 0.36.

The result is useful as directional decision-support for prioritizing pages for review, rather than as a causal explanation of why a page declined.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Error analysis for the Random Forest

rf_model = trained_models["Random Forest"]

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

rf_pred = (
    rf_prob >= 0.5
).astype(int)

error_frame = frame.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "is_declining_label",
    ]
].copy()

error_frame["model_probability"] = rf_prob
error_frame["prediction"] = rf_pred

error_frame["error_type"] = np.select(
    [
        (
            (error_frame["is_declining_label"] == 1)
            & (error_frame["prediction"] == 0)
        ),

        (
            (error_frame["is_declining_label"] == 0)
            & (error_frame["prediction"] == 1)
        ),
    ],
    [
        "False Negative",
        "False Positive",
    ],
    default="Correct",
)

print("Error counts:")
display(
    error_frame["error_type"]
    .value_counts()
)

print("\nThree false negatives:")
display(
    error_frame[
        error_frame["error_type"] == "False Negative"
    ]
    .sort_values("model_probability")
    .head(3)
)

print("\nThree false positives:")
display(
    error_frame[
        error_frame["error_type"] == "False Positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(3)
)

# Feature importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop 10 Random Forest features:")
display(
    importance.head(10)
)

Error counts:


,count
error_type,
Correct,1563
False Positive,529
False Negative,233



Three false negatives:


,content_id,client_id,is_declining_label,model_probability,prediction,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,1,0.079867,0,False Negative
3879,content_34b14c00f80c,client_d4735e3a26,1,0.082196,0,False Negative
27177,content_79ac977c6e0b,client_f74efabef1,1,0.149546,0,False Negative



Three false positives:


,content_id,client_id,is_declining_label,model_probability,prediction,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,0.737130,1,False Positive
23559,content_00603b0349b4,client_f74efabef1,0,0.734944,1,False Positive
25913,content_331182ca4cae,client_f74efabef1,0,0.733631,1,False Positive



Top 10 Random Forest features:


,feature,importance
9,days_with_impressions,0.134951
5,log_impressions_90d,0.129377
14,avg_position,0.109203
11,content_age_days,0.092048
4,char_count,0.038676
32,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
3,word_count,0.035406
13,ctr,0.035156
16,scroll_rate,0.033876


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.